In [ ]:
# rflib permite 
from rdflib import Graph, URIRef, Literal, Namespace, BNode
from rdflib.namespace import FOAF, RDF, DCTERMS, VOID, DC, SKOS, OWL, XSD
import pandas as pd

In [6]:
df = pd.read_csv("data/incendios-filtrado.csv")

In [7]:
SCHEMA = Namespace("https://schema.org/")
DCMI = Namespace("http://purl.org/dc/dcmitype/")
GEO = Namespace("http://www.opengis.net/ont/geosparql#")
WGS = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")

BASE = Namespace("http://example.org/event/")

In [8]:
g = Graph()
g.bind("schema", SCHEMA)
g.bind("dcmitype", DCMI)
g.bind("geo", GEO)
g.bind("wgs", WGS)
g.bind("owl", OWL)
g.bind("base", BASE)

In [9]:
g.add((SCHEMA.Event, OWL.equivalentClass, DCMI.Event))
g.add((SCHEMA.Place, OWL.equivalentClass, GEO.Feature))
g.add((SCHEMA.GeoCoordinates, OWL.equivalentClass, WGS.Point))

<Graph identifier=N5460f86699594629acefaff4b0f10322 (<class 'rdflib.graph.Graph'>)>

In [13]:
df = pd.read_csv("data/incendios-filtrado.csv", sep=";")


In [14]:
for index, row in df.iterrows():
    # URIs
    event_uri = URIRef(f"{BASE}event/{row['id']}")
    place_uri = URIRef(f"{BASE}place/{row['id']}")
    geo_bnode = BNode()

    # ----- PLACE -----
    g.add((place_uri, RDF.type, SCHEMA.Place))
    g.add((place_uri, SCHEMA.name, Literal(f"Lugar del incidente {row['id']}", lang="es")))
    g.add((place_uri, SCHEMA.geo, geo_bnode))

    # GeoCoordinates
    g.add((geo_bnode, RDF.type, SCHEMA.GeoCoordinates))
    if pd.notna(row.get("lat")):
        g.add((geo_bnode, SCHEMA.latitude, Literal(float(row["lat"]), datatype=XSD.float)))
    if pd.notna(row.get("long")):
        g.add((geo_bnode, SCHEMA.longitude, Literal(float(row["long"]), datatype=XSD.float)))

    # ----- EVENT -----
    g.add((event_uri, RDF.type, SCHEMA.Event))

    # --- about ---
    if "titulo" in df.columns and pd.notna(row.get("titulo")):
        g.add((event_uri, SCHEMA.about, Literal(row["titulo"], lang="es")))

    # --- description ---
    if "descripcion" in df.columns and pd.notna(row.get("descripcion")):
        g.add((event_uri, SCHEMA.description, Literal(row["descripcion"], lang="es")))

    # --- startDate ---
    if "fecha_inicio" in df.columns and pd.notna(row.get("fecha_inicio")):
        fecha_iso = pd.to_datetime(row["fecha_inicio"]).isoformat()
        g.add((event_uri, SCHEMA.startDate, Literal(fecha_iso, datatype=XSD.dateTime)))

    # --- location (points to Place) ---
    g.add((event_uri, SCHEMA.location, place_uri))

In [16]:
output_path = "data/incendios.ttl"
g.serialize(destination=output_path, format="turtle")

print("RDF generado usando solo las propiedades solicitadas:", output_path)

RDF generado usando solo las propiedades solicitadas: data/incendios.ttl
